# 4. Attention as an exact lookup

**The question:** to write `S`, the model needs the byte `H`, 2 tokens back. How does attention find exactly that
token?

An attention head compares the current token's **query** with every token's **key**. The weights here make the query
name a position and the key state a position, so only one token matches.

In [1]:
import numpy as np

import helpers
from helpers import attend, load, named, rows, shown, stream

model = load('base64')
r = model.r
tokens = stream(model)
X = rows(model, tokens[:4])              # ENC H e l : the model is about to write S
print([shown(t) for t in tokens[:4]])
head = next(h for h in r.heads if h.name == 'enc_first')
print('Q', head.Q.shape, 'K', head.K.shape, 'V', head.V.shape, 'O', head.O.shape)

['ENC', "x48 'H'", "x65 'e'", "x6c 'l'"]
Q (76, 9) K (76, 9) V (76, 10) O (10, 76)


## 1. The query

The token predicting `S` is `l`, at phase 3. The distance table says `first` is 2 back, so at phase 3 the head asks for
phase 1. That is one entry in Q:

In [2]:
q = X[-1] @ head.Q
print('query (one number per phase, then a fallback):')
print(q)
print('Q row for phase[3]:', head.Q[r.S.at('phase', 3)])

query (one number per phase, then a fallback):
[0.  1.  0.  0.  0.  0.  0.  0.  0.5]
Q row for phase[3]: [0. 1. 0. 0. 0. 0. 0. 0. 0.]


The 1 is at index 1: "phase 1". The last number, 0.5, is a fallback explained below.

## 2. The keys

Each token's key states its own phase. The mode token also matches the fallback.

In [3]:
keys = X @ head.K
for t, k in zip(tokens[:4], keys):
  print(f'{shown(t):<10} key {k}')

ENC        key [1. 0. 0. 0. 0. 0. 0. 0. 1.]
x48 'H'    key [0. 1. 0. 0. 0. 0. 0. 0. 0.]
x65 'e'    key [0. 0. 1. 0. 0. 0. 0. 0. 0.]
x6c 'l'    key [0. 0. 0. 1. 0. 0. 0. 0. 0.]


## 3. Scores and weights

The score is query times key, multiplied by 100 so that the winner takes all of the softmax.

In [4]:
scores, weights, added = attend(model, X, head)
for t, s, w in zip(tokens[:4], scores, weights):
  print(f'{shown(t):<10} score {s:6.1f}   weight {w:.20f}')

ENC        score   50.0   weight 0.00000000000000000000
x48 'H'    score  100.0   weight 1.00000000000000000000
x65 'e'    score    0.0   weight 0.00000000000000000000
x6c 'l'    score    0.0   weight 0.00000000000000000000


`H` scores 100, `ENC` scores 50 through the fallback, and the rest 0. The softmax puts all the weight on `H`. If no
token were at the asked position, `ENC` would win, and that reads as "missing".

## 4. What it brings

V and O copy the token's bits into the head's own named block.

In [5]:
print(named(model, added))

{'enc_first.bits[3]': 1.0, 'enc_first.bits[6]': 1.0}


Bits 3 and 6 of `H` are now in `enc_first.bits`. The second head does the same for `e`, 1 back:

In [6]:
second = next(h for h in r.heads if h.name == 'enc_second')
s2, w2, added2 = attend(model, X, second)
print('reads', shown(tokens[int(np.argmax(w2))]))
print(named(model, added2))

reads x65 'e'
{'enc_second.bits[0]': 1.0, 'enc_second.bits[2]': 1.0, 'enc_second.bits[5]': 1.0, 'enc_second.bits[6]': 1.0}


`e` is `0x65`, binary `01100101`: bits 0, 2, 5 and 6.

There is also a `mode` head, which finds the group's `ENC` token and sets `mode_enc`. Together:

In [7]:
x = X[-1].copy()
for h in r.heads:
  x = x + attend(model, X, h)[2]
print(named(model, x))

{'one': 1.0, 'bits[2]': 1.0, 'bits[3]': 1.0, 'bits[5]': 1.0, 'bits[6]': 1.0, 'is_byte': 1.0, 'phase[3]': 1.0, 'mode_enc': 1.0, 'dec_first.missing': 1.0, 'dec_second.missing': 1.0, 'enc_first.bits[3]': 1.0, 'enc_first.bits[6]': 1.0, 'enc_second.bits[0]': 1.0, 'enc_second.bits[2]': 1.0, 'enc_second.bits[5]': 1.0, 'enc_second.bits[6]': 1.0}


This vector is what the MLP receives: the token `l`, and on top of it what attention brought. The two decoding heads
run too, find nothing to read, and set `dec_first.missing` and `dec_second.missing`. Nothing uses those while encoding,
because every encoding unit also needs `mode_enc`.